# e2m tutorial: TCGA LUAD

This tutorial follows the manuscript workflow for one cancer type using the **object API**:
a `Dataset` holds the aligned expression / mutation / TMB tables, and an `E2MModel` (multitask mutation network) or `TmbModel` (tree regressor) is fitted on it and returns pandas directly. The final **Appendix** shows the equivalent functional API and CLI, which write the same on-disk artifacts and are what the `e2m` command runs.

Install and run from the repository root:

```bash
pip install -e ".[interpretation]"
```

The training and SHAP cells can take a few minutes on CPU.

In [ ]:
from pathlib import Path
import pandas as pd

import e2m
from e2m import Dataset, E2MModel, TmbModel

e2m.set_verbose()  # print progress to stderr; call e2m.set_verbose(False) to silence

CANCERS = ["LUAD"]
DATA_DIR = Path("e2m_data")
MODEL_DIR = Path("models/luad")
RESULT_DIR = Path("results/luad")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

## 0. Discover the API

Before running anything: `e2m.format_options()` lists every accepted choice, and each object's docstring describes its parameters and methods. On the command line the same help is `e2m --help`, `e2m <command> --help`, and `e2m options`.

In [ ]:
print(e2m.format_options())   # expression datasets, transforms, TMB models, SHAP methods, cancers

# programmatic access, e.g. when building data_overrides:
e2m.options()["expression_transform"]

In [ ]:
import re

def _clean(text):  # drop reST markup so docstrings read cleanly
    return re.sub(r":\w+:`([^`]*)`", r"\1", text).replace("``", "")

# The model docstring plus a one-line summary of each public method
# (cleaner than help(), which also prints inherited dunder methods).
print(_clean(E2MModel.__doc__))
for _name in ("fit", "predict", "cross_validate", "embed", "head_weights", "save", "load"):
    _summary = (getattr(E2MModel, _name).__doc__ or "").strip().splitlines()
    print(f"  .{_name + '()':17s}{_clean(_summary[0]) if _summary else ''}")

In [ ]:
import sys, subprocess
# a real CLI help run (the object API mirrors these commands). On the command line this
# is simply `e2m --help`; here we call it through the current interpreter to be portable.
print(subprocess.run([sys.executable, "-m", "e2m", "--help"], capture_output=True, text=True).stdout)

## 1. Build a `Dataset`

`Dataset.from_tcga` downloads cohort-level Xena STAR counts and MC3 mutation data, converts the Xena log2 values back to counts and applies `log1p`, keeps GENCODE v36 protein-coding genes, maps Ensembl ids to symbols, and matches expression to mutations by sample. Mutation targets present in at least 5 percent of samples are kept, capped at the 400 most frequent. Everything is cached under `DATA_DIR`.

In [ ]:
data = Dataset.from_tcga(
    CANCERS,
    data_dir=DATA_DIR,
    data_overrides={"expression_dataset": "star_counts", "expression_transform": "log1p"},
)
data   # Dataset(samples=..., genes=..., targets=..., tmb=True, cohorts=['LUAD'])

In [ ]:
print("samples:", len(data.samples))
print("genes:  ", len(data.genes))
print("targets:", len(data.targets))

# most frequently mutated targets in this cohort
data.mutations.mean().sort_values(ascending=False).head(10)

## 2. Configure a model explicitly

`E2MModel` takes the network hyperparameters as keyword arguments; anything you omit falls back to the packaged manuscript defaults. The values below are those defaults, written out so you can see and change them. The shared encoder learns a sample representation, and one sigmoid output predicts each mutation target.

In [ ]:
model = E2MModel(
    hidden_layers=[512, 256],   # shared-encoder widths
    dropout_rate=0.3,
    learning_rate=5e-4,
    weight_decay=3e-4,
    batch_size=64,
    epochs=100,                 # lower to ~20 for a quick tutorial pass
    patience=6,                 # early stopping on an internal validation split
)
model   # E2MModel(unfitted, targets=0)

## 3. Held-out cross-validation

`cross_validate` runs five-fold, cohort-stratified CV with the model's settings and returns one metrics row per target. It does not modify `model` (each fold trains its own network); use it to estimate performance. Normalized AUPRC is 0 at the prevalence baseline and 1 for perfect ranking. Pass `output=...` to also write the out-of-fold tables to disk.

In [ ]:
metrics = model.cross_validate(data)
metrics.sort_values("normalized_auprc", ascending=False).head(15)

## 4. Fit on a training split and predict held-out samples

`subset` slices a `Dataset` by samples (or targets). Here we hold out samples explicitly, fit on the rest, and predict. `predict` returns a samples-by-targets probability frame.

In [ ]:
train = data.subset(samples=data.samples[:400])
test = data.subset(samples=data.samples[400:])

model.fit(train)                       # trains this model on the 400 training samples
probabilities = model.predict(test)    # held-out probabilities
probabilities.iloc[:5, :6]

In [ ]:
# predict() also accepts a plain samples-by-genes DataFrame (e.g. an external cohort);
# input genes are aligned to the training features by symbol.
model.predict(test.expression.iloc[:3]).iloc[:, :6]

## 5. Embeddings and output-head weights

The shared encoder gives a 256-dimensional representation per sample (useful for clustering or visualization); each output head is one weight vector per target.

In [ ]:
embeddings = model.embed(data)     # samples x 256
weights = model.head_weights()     # targets x 256
embeddings.shape, weights.shape

### Optional: UMAP + Leiden on the embeddings and head weights

The 256-dimensional embeddings (one point per sample) and head weights (one point per mutation target) are easy to view in 2D. This step is optional and needs extra packages:

```bash
pip install umap-learn leidenalg igraph matplotlib
```

The cells below import them lazily and skip themselves (printing a note) if the packages are not installed, so the rest of the tutorial still runs. As in the manuscript figures, points are colored by Leiden clusters computed on UMAP's own weighted graph.

In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    import umap
    import igraph as ig
    import leidenalg
    from sklearn.preprocessing import StandardScaler
    HAS_VIZ = True
except ImportError as exc:
    HAS_VIZ = False
    print(f"Skipping UMAP/Leiden viz — install umap-learn, leidenalg, igraph, matplotlib. ({exc})")


def umap_leiden(matrix, n_neighbors=15, min_dist=0.3, resolution=1.0, seed=0):
    """UMAP 2D coords + Leiden clusters on UMAP's own weighted graph (as in the paper's figures)."""
    values = StandardScaler().fit_transform(np.asarray(matrix, dtype=float))
    n_neighbors = max(2, min(n_neighbors, values.shape[0] - 1))
    reducer = umap.UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist,
                        metric="euclidean", random_state=seed)
    coords = reducer.fit_transform(values)
    graph = reducer.graph_.tocoo()
    g = ig.Graph(n=values.shape[0], edges=list(zip(graph.row.tolist(), graph.col.tolist())), directed=False)
    g.es["weight"] = graph.data.tolist()
    part = leidenalg.find_partition(g, leidenalg.RBConfigurationVertexPartition,
                                    weights=g.es["weight"], resolution_parameter=resolution, seed=seed)
    return coords, np.asarray(part.membership)


def plot_umap(matrix, title, n_neighbors=15, resolution=1.0, cmap="tab10", size=16):
    coords, clusters = umap_leiden(matrix, n_neighbors=n_neighbors, resolution=resolution)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    ax.scatter(coords[:, 0], coords[:, 1], c=clusters, cmap=cmap, s=size, linewidths=0)
    ax.set_title(f"{title} — {clusters.max() + 1} Leiden clusters")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    ax.set_xticks([])
    ax.set_yticks([])
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.tight_layout()
    plt.show()
    return clusters

In [ ]:
if HAS_VIZ:
    sample_clusters = plot_umap(embeddings.values, "LUAD samples — embedding UMAP", n_neighbors=15)

In [ ]:
if HAS_VIZ:
    target_clusters = plot_umap(weights.values, "Mutation targets — head-weight UMAP",
                                n_neighbors=10, cmap="tab20", size=28)

## 6. Save the model bundle

Persist the fitted model. The bundle is what `E2MModel.load(...)` reads, and also what the SHAP step and the CLI `predict` / `embed` commands consume.

In [ ]:
model.save(MODEL_DIR)
sorted(p.name for p in MODEL_DIR.iterdir())

## 7. Interpret one target with SHAP

SHAP interpretation runs on the saved bundle through `e2m.explain`. The default `method="xgboost"` trains a separate XGBoost classifier for the target and applies Tree SHAP — the manuscript interpretation step. The attribution model is pluggable (`lightgbm`, `random_forest`, `gradient_boosting`), and `method="neural"` explains the multitask network output itself. The attribution model is never the cross-validated predictor from step 3.

In [ ]:
xgb_shap = e2m.explain(
    MODEL_DIR, target="TP53", method="xgboost",
    output_dir=RESULT_DIR / "shap_xgboost_tp53", data_dir=DATA_DIR,
)
pd.read_csv(RESULT_DIR / "shap_xgboost_tp53/feature_summary.csv").head(15)

In [ ]:
# Explain the neural output directly (gradient SHAP on the TP53 probability).
e2m.explain(
    MODEL_DIR, target="TP53", method="neural",
    output_dir=RESULT_DIR / "shap_neural_tp53", data_dir=DATA_DIR,
)
pd.read_csv(RESULT_DIR / "shap_neural_tp53/feature_summary.csv").head(15)

## 8. TMB prediction with `TmbModel`

`TmbModel` predicts `log2(coding TMB + 1)` with a pluggable tree ML model. Samples without a coding MC3 event are dropped. `cross_validate` returns per-cancer and overall Spearman / Pearson / MAE / RMSE; `fit` + `predict` deploy it.

In [ ]:
tmb_model = TmbModel(ml_model="xgboost", n_estimators=300, learning_rate=0.1)
tmb_summary = tmb_model.cross_validate(data)
tmb_summary

In [ ]:
tmb_model.fit(data)
tmb_model.predict(data.subset(samples=data.samples[:5]))
# other models: TmbModel(ml_model="random_forest" | "gradient_boosting" | "lightgbm")

## Appendix. The same workflow on the command line

The object API above is the recommended way to work interactively. The identical workflow is available through the `e2m` command line, which writes the standardized manuscript artifacts to disk (`metrics.csv`, `oof_probabilities.csv`, `summary.csv`, `run_metadata.json`, and a model bundle). Every command accepts `--verbose`.

```bash
e2m options
e2m download --cancer LUAD --transform log1p --data-dir ./e2m_data --output results/luad/data
e2m cv       --cancer LUAD --transform log1p --data-dir ./e2m_data --output results/luad/mutation_cv
e2m tmb      --cancer LUAD --transform log1p --data-dir ./e2m_data --output results/luad/tmb_cv
e2m train    --cancer LUAD --transform log1p --data-dir ./e2m_data --output models/luad
e2m explain  --model-dir models/luad --target TP53 --method xgboost \
             --data-dir ./e2m_data --output results/luad/shap_xgboost_tp53
```

The CLI is driven by a stateless functional API (`e2m.cross_validate`, `e2m.train`, ...) that is also importable if you want the file-writing behavior from Python. See [external_transfer.ipynb](external_transfer.ipynb) for predicting on an external GEO cohort with a TCGA-trained model.

## Interpretation note

SHAP reports features used by a model. In bulk RNA data, those features can reflect mutation-associated expression, subtype, co-mutation, immune cells, stromal cells, or other correlated biology. Do not treat a SHAP association as proof of a direct causal effect.